# First Diagnostic: `count_common` Changes Along a Route

This notebook checks whether, for the same `month`, `DayOfWeek`, and `HourSourceTime`, the value of `count_common` changes across stations on the same `route_id`.

A change in adjacent stations can indicate potential station skipping behavior.

In [1]:
import pandas as pd
from pathlib import Path

# Load the merged route-level dataset
csv_path = Path('..') / 'govData' / 'ride_data_merged.csv'
df = pd.read_csv(csv_path)

# Keep only columns needed for this first diagnostic
cols = [
    'month', 'route_id', 'DayOfWeek', 'HourSourceTime',
    'StopSequence_Rishui', 'StopCode', 'count_common'
]
work = df[cols].copy()

# Ensure numeric columns are correctly typed
for c in ['month', 'DayOfWeek', 'HourSourceTime', 'StopSequence_Rishui', 'count_common']:
    work[c] = pd.to_numeric(work[c], errors='coerce')

# Drop rows missing key fields
work = work.dropna(subset=['month', 'route_id', 'DayOfWeek', 'HourSourceTime', 'StopSequence_Rishui', 'count_common'])

print(f'Loaded rows for analysis: {len(work):,}')
work.head()

Loaded rows for analysis: 225,802


,month,route_id,DayOfWeek,HourSourceTime,StopSequence_Rishui,StopCode,count_common
0,3,37936,1,15,2,3301,39
1,3,37936,1,15,3,5987,39
2,3,37936,1,15,4,5988,39
3,3,37936,1,15,5,9921,39
4,3,37936,1,15,6,2540,39


In [3]:
# Group definition requested: same month + day of week + hour on the same route
key = ['route_id', 'month', 'DayOfWeek', 'HourSourceTime']

# 1) Which groups have inconsistent count_common across stations?
group_check = (
    work.groupby(key)
    .agg(
        stations_in_group=('StopCode', 'nunique'),
        unique_count_common=('count_common', 'nunique')
    )
    .reset_index()
)

inconsistent_groups = group_check[group_check['unique_count_common'] > 1].copy()

print(f'Groups checked: {len(group_check):,}')
print(f'Groups with count_common changes across stations: {len(inconsistent_groups):,}')
display(inconsistent_groups.sort_values('unique_count_common', ascending=False).head(20))

# 2) Detect adjacent-station transitions where count_common changes
ordered = work.sort_values(key + ['StopSequence_Rishui'])
ordered['prev_count_common'] = ordered.groupby(key)['count_common'].shift(1)
ordered['prev_stop_code'] = ordered.groupby(key)['StopCode'].shift(1)
ordered['prev_stop_seq'] = ordered.groupby(key)['StopSequence_Rishui'].shift(1)

changes = ordered[
    ordered['prev_count_common'].notna() &
    (ordered['count_common'] != ordered['prev_count_common'])
].copy()

changes['delta_count_common'] = changes['count_common'] - changes['prev_count_common']

# Filter to stronger changes only
changes = changes[changes['delta_count_common'].abs() >= 3].copy()

result_cols = [
    'route_id', 'month', 'DayOfWeek', 'HourSourceTime',
    'prev_stop_seq', 'StopSequence_Rishui',
    'prev_stop_code', 'StopCode',
    'prev_count_common', 'count_common', 'delta_count_common'
]

print(f'Adjacent station transitions with abs(delta_count_common) >= 3: {len(changes):,}')
display(changes[result_cols].head(50))

# Optional compact summary: how often each route shows these transitions
route_summary = (
    changes.groupby('route_id')
    .size()
    .rename('num_changed_transitions')
    .sort_values(ascending=False)
    .reset_index()
)

display(route_summary.head(20))

Groups checked: 4,315
Groups with count_common changes across stations: 169


,route_id,month,DayOfWeek,HourSourceTime,stations_in_group,unique_count_common
1866,10802,1,2,24,42,3
1867,10802,1,2,25,42,3
179,5499,1,1,13,55,2
177,5499,1,1,11,55,2
182,5499,1,1,16,55,2
187,5499,1,2,5,55,2
188,5499,1,2,6,55,2
181,5499,1,1,15,55,2
173,5499,1,1,7,55,2
191,5499,1,2,9,55,2


Adjacent station transitions with abs(delta_count_common) >= 3: 6,054


,route_id,month,DayOfWeek,HourSourceTime,prev_stop_seq,StopSequence_Rishui,prev_stop_code,StopCode,prev_count_common,count_common,delta_count_common
129483,5499,1,1,16,1.0,1,3324.0,3324,12.0,16,4.0
78002,5499,1,1,16,1.0,2,3324.0,6268,16.0,12,-4.0
129484,5499,1,1,16,2.0,2,6268.0,6268,12.0,16,4.0
78003,5499,1,1,16,2.0,3,6268.0,9907,16.0,12,-4.0
129485,5499,1,1,16,3.0,3,9907.0,9907,12.0,16,4.0
78004,5499,1,1,16,3.0,4,9907.0,9908,16.0,12,-4.0
129486,5499,1,1,16,4.0,4,9908.0,9908,12.0,16,4.0
78005,5499,1,1,16,4.0,5,9908.0,3097,16.0,12,-4.0
129487,5499,1,1,16,5.0,5,3097.0,3097,12.0,16,4.0
78006,5499,1,1,16,5.0,6,3097.0,2181,16.0,12,-4.0


,route_id,num_changed_transitions
0,10802,2842
1,5499,2208
2,10398,1004


In [4]:
# 3) How common are duplicate rows for the same station on the same route and day?
# Note: this dataset has month + day-of-week (not a full calendar date),
# so "same day" is measured as (month, DayOfWeek).

# A) Same route + station + day-bucket
key_day = ['route_id', 'StopCode', 'month', 'DayOfWeek']
count_day = (
    work.groupby(key_day)
    .size()
    .rename('rows_per_key')
    .reset_index()
)

dup_day = count_day[count_day['rows_per_key'] > 1].copy()

pct_dup_day = (len(dup_day) / len(count_day) * 100) if len(count_day) else 0

print('Duplicate prevalence (same route + station + month + day-of-week):')
print(f'  unique keys checked: {len(count_day):,}')
print(f'  keys with >1 row:    {len(dup_day):,}')
print(f'  prevalence:          {pct_dup_day:.2f}%')
display(dup_day.sort_values('rows_per_key', ascending=False).head(20))

# B) Stricter check: same route + station + day-bucket + hour
key_day_hour = ['route_id', 'StopCode', 'month', 'DayOfWeek', 'HourSourceTime']
count_day_hour = (
    work.groupby(key_day_hour)
    .size()
    .rename('rows_per_key')
    .reset_index()
)

dup_day_hour = count_day_hour[count_day_hour['rows_per_key'] > 1].copy()

pct_dup_day_hour = (len(dup_day_hour) / len(count_day_hour) * 100) if len(count_day_hour) else 0

print('\nDuplicate prevalence (same route + station + month + day-of-week + hour):')
print(f'  unique keys checked: {len(count_day_hour):,}')
print(f'  keys with >1 row:    {len(dup_day_hour):,}')
print(f'  prevalence:          {pct_dup_day_hour:.2f}%')
display(dup_day_hour.sort_values('rows_per_key', ascending=False).head(20))

Duplicate prevalence (same route + station + month + day-of-week):
  unique keys checked: 15,211
  keys with >1 row:    14,710
  prevalence:          96.71%


,route_id,StopCode,month,DayOfWeek,rows_per_key
10049,10802,2024,1,2,67
10050,10802,2024,1,3,67
8698,10802,1080,1,3,67
8539,10802,1054,1,2,67
8540,10802,1054,1,3,67
8620,10802,1079,1,3,67
11090,10802,4173,1,3,67
9840,10802,1575,1,3,67
10129,10802,2040,1,2,67
10850,10802,3230,1,3,67



Duplicate prevalence (same route + station + month + day-of-week + hour):
  unique keys checked: 181,878
  keys with >1 row:    33,945
  prevalence:          18.66%


,route_id,StopCode,month,DayOfWeek,HourSourceTime,rows_per_key
65284,10398,1370,1,4,12,7
64374,10398,1367,1,4,12,7
64828,10398,1368,1,4,12,7
71163,10398,2024,1,1,12,6
75745,10398,3379,1,1,12,6
72086,10398,2410,1,1,12,6
61718,10398,264,1,4,12,6
64797,10398,1368,1,1,12,6
66619,10398,1444,1,1,12,6
62175,10398,427,1,4,12,6


In [ ]:
# 4) Strict duplicate check: same route + station + day-bucket + hour + stop sequence
key_day_hour_seq = [
    'route_id', 'StopCode', 'month', 'DayOfWeek', 'HourSourceTime', 'StopSequence_Rishui'
]

count_day_hour_seq = (
    work.groupby(key_day_hour_seq)
    .size()
    .rename('rows_per_key')
    .reset_index()
)

dup_day_hour_seq = count_day_hour_seq[count_day_hour_seq['rows_per_key'] > 1].copy()

pct_dup_day_hour_seq = (len(dup_day_hour_seq) / len(count_day_hour_seq) * 100) if len(count_day_hour_seq) else 0

print('Duplicate prevalence (same route + station + month + day-of-week + hour + stop sequence):')
print(f'  unique keys checked: {len(count_day_hour_seq):,}')
print(f'  keys with >1 row:    {len(dup_day_hour_seq):,}')
print(f'  prevalence:          {pct_dup_day_hour_seq:.2f}%')

display(dup_day_hour_seq.sort_values('rows_per_key', ascending=False).head(20))